# Batch-correction API examples

This notebook demonstrates the two public batch-correction interfaces:

- `scbatch.pp.batch_correct`, the Scanpy-style functional interface.
- `scbatch.pp.BatchCorrectionPredictor`, the future fitted-predictor interface.

The currently registered methods are scaffolds. `batch_correct` prints an unsupported-operation message and returns the same `AnnData` object. Predictor fitting and transformation are not implemented yet.

In [33]:
from pathlib import Path
import sys

import numpy as np
from anndata import AnnData


# def _find_batch_correction_worktree() -> Path:
#     """Find the checkout that contains this example's API."""
#     current_directory = Path.cwd().resolve()
#     candidates = [current_directory, *current_directory.parents]
#     candidates.extend(
#         sibling
#         for directory in current_directory.parents
#         for sibling in directory.glob("scbatch-*")
#     )
#     for candidate in candidates:
#         if (candidate / "scbatch" / "pp" / "batch_correction").is_dir():
#             return candidate
#     raise ImportError(
#         "This example requires a checkout containing scbatch.pp.batch_correction."
#     )


worktree_root = "/Users/yishai/Desktop/build_pip_package/scbatch-basic-structure-creation"
sys.path.insert(0, str(worktree_root))
# sys.modules.pop("scbatch.pp", None)
# sys.modules.pop("scbatch", None)

from scbatch.pp import (
    BatchCorrectionPredictor,
    available_batch_correction_methods,
    batch_correct,
)

## Create a small annotated dataset

The public APIs use `batch_key`, `anchor_key`, and `layer` names. This small example includes each required location.

In [34]:
adata = AnnData(X=np.array([[1.0, 2.0], [3.0, 4.0]]))
adata.obs["batch"] = ["reference", "query"]
adata.obs["is_anchor"] = [True, False]
adata.layers["arcsinh"] = adata.X.copy()

adata
adata.obs

import pandas as pd
import anndata as ad
import numpy as np

# adata.X has 5 cells (rows) and 3 markers (columns)
X = np.zeros((100, 3))

# adata.var has 3 rows (matching the 3 columns of adata.X) and 4 columns of metadata
var_df = pd.DataFrame(
    {
        "gene_symbol": ["CD4", "CD8A", "CD19"],
        "chromosome": ["chr12", "chr2", "chr16"],
        "marker_type": ["surface", "surface", "surface"],
        "expressed": [True, True, False],
    },
    index=["gene_1", "gene_2", "gene_3"],
)

adata = ad.AnnData(X=X, var=var_df)

print(adata.var.shape)  # (3, 4) -> 3 rows (markers), 4 columns (metadata)
print(adata.var)

(3, 4)
       gene_symbol chromosome marker_type  expressed
gene_1         CD4      chr12     surface       True
gene_2        CD8A       chr2     surface       True
gene_3        CD19      chr16     surface      False


## Scanpy-style functional API

List the registered method keys, then select `qq` through `batch_correct`. The assertion documents the current pass-through behavior.

In [35]:
available_batch_correction_methods()

('boundary_extrapolated_qq',
 'qq',
 'cytonorm',
 'cycombine',
 'mmd_resnet',
 'harmony',
 'fastmnn')

In [36]:
corrected_adata = batch_correct(
    adata,
    method="qq",
    layer="arcsinh",
)

assert corrected_adata is adata
corrected_adata

scbatch.pp.batch_correct_qq is not implemented; returning AnnData unchanged.


AnnData object with n_obs × n_vars = 100 × 3
    var: 'gene_symbol', 'chromosome', 'marker_type', 'expressed'

## Predictor configuration API

Use `BatchCorrectionPredictor` to validate and retain a method selection for a future fitted workflow. `fit`, `transform`, and `fit_transform` intentionally raise `NotImplementedError` in the current scaffold, so this notebook does not call them.

In [37]:
predictor = BatchCorrectionPredictor(
    method="qq",
    batch_key="batch",
    anchor_key="is_anchor",
)

assert predictor.config.method == "qq"
assert predictor.available_batch_correction_methods() == available_batch_correction_methods()
predictor.config

BatchCorrectionConfig(method='qq', batch_key='batch', anchor_key='is_anchor', layer='arcsinh')